# CMSE 202 - Pneumonia Classification Using Traditional Machine Learning

## Multiclassifier Attempt

Imports and deleting .DS_Store files for macOS users

In [51]:
import os
from skimage.io import imread
from skimage.transform import resize
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


#Cleans up macOS .DS_Store files that can break directory loops
for root, dirs, files in os.walk(".."):
    for file in files:
        if file == ".DS_Store":
            os.remove(os.path.join(root, file))

## Loading in data and organizing it

In [43]:
# NOTE:
# This notebook is located in code/notebooks/
# It loads images from ../../data/processed/train and test
# Do NOT change the folder structure, or the relative paths will break!

target_class = "Pneumonia"  # Change this to whatever class you want
data_root = os.path.join("..", "..", "data", "processed")
image_shape = (64, 64, 3)

# This implementation was developed with assistance from ChatGPT (OpenAI, 2025).
# ChatGPT helped to load in the data and classify it


def load_binary_data(split_dir, target_class, image_shape):
    """
    Loads image data from a directory and converts it into feature vectors and binary labels.

    Parameters:
    - split_dir (str): Path to the folder containing class subfolders.
    - target_class (str): The name of the class to label as 1. All other classes will be labeled as 0.
    - image_shape (tuple): Desired shape for each image (e.g., (64, 64, 3)).

    Returns:
    - X (np.array): Flattened image data for each image.
    - y (np.array): Binary labels (1 = target class, 0 = all others).
    """
    
    X = []
    y = []

    # Loops through each subfolder in the given directory
    for label in os.listdir(split_dir):
        label_path = os.path.join(split_dir, label)

        # Skips system files like .DS_Store or anything that's not a folder
        if not os.path.isdir(label_path):
            continue

        # Determines if this subfolder corresponds to the target class
        is_target = (label == target_class)

        # Loops through each image file in the subfolder
        for fname in os.listdir(label_path):
            img_path = os.path.join(label_path, fname)

            # Skips hidden/system files or other non-image files
            if not os.path.isfile(img_path) or fname.startswith("."):
                continue

            try:
                # Load and resize the image to the specified shape
                img = imread(img_path)
                img_resized = resize(img, image_shape)

                # Flatten the resized image into a 1D feature vector
                X.append(img_resized.flatten())

                # Append the label: 1 for target class, 0 for others
                y.append(1 if is_target else 0)

            except Exception as e:
                # If an image fails to load or process, print a warning and skip it
                print(f"⚠️ Skipping {img_path}: {e}")
    
    return np.array(X), np.array(y)

## Splitting the data

In [35]:
#Lables training and testing data
train_path = os.path.join(data_root, "train")
test_path = os.path.join(data_root, "test")
#Apply the function above tho the data
train_vectors, train_labels = load_binary_data(train_path, target_class, image_shape)
test_vectors, test_labels = load_binary_data(test_path, target_class, image_shape)
print("Train vector shape:", train_vectors.shape)
print("Train labels shape:", train_labels.shape)

Train vector shape: (2800, 12288)
Train labels shape: (2800,)


## Train the model

In [48]:
#Feel free to change the training part however you want, whatever disease you want to use you can just go replace target_class = "Pneumonia" with that.
train_vectors = train_vectors / 255.0 #normalizing the pixel values
print("Train vectors normalized")

test_vectors = test_vectors / 255.0
print("Test vectors normalized")

print("Fitting the classifier to the training set")
start = time.time() #Start timing

clf = OneVsRestClassifier(SVC(kernel='linear', class_weight='balanced'))
clf = clf.fit(train_vectors, train_labels)

# print("Best estimator found by grid search:")
# print(clf.best_estimator_)
# print("Best parameters found by grid search:")
# print(clf.best_params_)

end = time.time()

print("Training complete.")
print("Runtime:", end - start)



Train vectors normalized
Test vectors normalized
Fitting the classifier to the training set
Training complete.
Runtime: 77.2741470336914


## Test the model

In [53]:
predict_vectors = test_vectors #setting up labels to differentiate from training and testing
true_labels = test_labels

print("Predictions based on training")
pred_labels = clf.predict(predict_vectors) #Using the predict function to create predictions 

print(classification_report(true_labels, pred_labels)) #Analyzing the predictions to see accuracy and results of predictions
print(confusion_matrix(true_labels, pred_labels))
print("Accuracy:", accuracy_score(true_labels, pred_labels))

Predictions based on training
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       650
           1       0.07      1.00      0.13        50

    accuracy                           0.07       700
   macro avg       0.04      0.50      0.07       700
weighted avg       0.01      0.07      0.01       700

[[  0 650]
 [  0  50]]
Accuracy: 0.07142857142857142


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
